# Classic SPL LTV Validation
Compares `classic-specialty-ltv` pipeline outputs (`sedo_exploration` branch) against ELTV v8.2 production baseline.

## Step 1: Setup

In [ ]:
%env ENV_FOR_DYNACONF = prod
%env DYNACONF_GIT_BRANCH = sedo_exploration

import numpy as np
import pandas as pd
import eltv.util.non_spark_helpers as nsh
from eltv import paths as p

nsh.initialize_config_path(p)

## Step 2: Load Classic SPL outputs (your run)

In [ ]:
df_classic = nsh.read_parquet_s3_to_pandas(
    "tmx-smsiweb/classic-specialty-ltv/prod/sedo_exploration/score/score_internal_results/"
)
df_classic = df_classic.query("release_day == 28")
print("Classic SPL row count:", len(df_classic))
df_classic.head()

## Step 3: Load ELTV v8.2 reference (answer key)

In [ ]:
df_eltv = nsh.read_parquet_s3_to_pandas(
    "tmx-smsiweb/eltv-policy/prod/v8.2/score/score_internal_results/"
)
df_eltv = df_eltv.query("release_day == 28")
print("ELTV v8.2 row count:", len(df_eltv))
df_eltv.head()

## Step 4: Select columns and merge on policy ID

In [ ]:
# Add balanced loss to ELTV (classic SPL loss is already balanced)
df_eltv["loss_bal"] = df_eltv["loss"] + df_eltv["lr_balance_amt"]

classic_cols = ['adw_pol_id', 'premium', 'loss', 'ple', 'ltv_num_e005scl', 'aac_mkt']
eltv_cols    = ['adw_pol_id', 'premium', 'loss_bal', 'ple', 'ltv_num_e005scl', 'aac_mkt_e005scl']

df_merged = df_classic[classic_cols].merge(
    df_eltv[eltv_cols],
    on='adw_pol_id',
    how='inner',
    suffixes=('_classic', '_eltv')
)
print("Merged row count:", len(df_merged))
df_merged.head()

## Step 5: Run np.isclose checks
All should return `True` to pass validation.

In [ ]:
checks = {
    "premium" : np.isclose(df_merged["premium_classic"],      df_merged["premium_eltv"],        equal_nan=True).all(),
    "loss"    : np.isclose(df_merged["loss_classic"],         df_merged["loss_bal"],             equal_nan=True).all(),
    "ple"     : np.isclose(df_merged["ple_classic"],          df_merged["ple_eltv"],             equal_nan=True).all(),
    "ltv_num" : np.isclose(df_merged["ltv_num_e005scl_classic"], df_merged["ltv_num_e005scl_eltv"], equal_nan=True).all(),
    "aac_mkt" : np.isclose(df_merged["aac_mkt"],              df_merged["aac_mkt_e005scl"],      equal_nan=True).all(),
}

for metric, result in checks.items():
    status = "✅ PASS" if result else "❌ FAIL"
    print(f"{metric:10s}: {status}")

## Step 6: Spot-check counts by line and release_day
If any checks above failed, start digging here.

In [ ]:
classic_agg = (
    df_classic
    .groupby(['drv_line', 'release_day'])
    .agg(count=('adw_pol_id', 'count'),
         avg_ple=('ple', 'mean'),
         avg_ltv=('ltv_num_e005scl', 'mean'),
         avg_aac=('aac_mkt', 'mean'))
    .reset_index()
)

eltv_agg = (
    df_eltv
    .groupby(['drv_line', 'release_day'])
    .agg(count=('adw_pol_id', 'count'),
         avg_ple=('ple', 'mean'),
         avg_ltv=('ltv_num_e005scl', 'mean'),
         avg_aac=('aac_mkt_e005scl', 'mean'))
    .reset_index()
)

comparison = classic_agg.merge(eltv_agg, on=['drv_line', 'release_day'], suffixes=('_classic', '_eltv'))
comparison['ple_diff']  = comparison['avg_ple_classic']  - comparison['avg_ple_eltv']
comparison['ltv_diff']  = comparison['avg_ltv_classic']  - comparison['avg_ltv_eltv']
comparison['count_diff'] = comparison['count_classic']   - comparison['count_eltv']
comparison